# MindForge Initiative
DeepSeek Chat History → Obsidian Exporter

Exports all DeepSeek web conversations to Markdown files with YAML frontmatter, ready for Obsidian import.

**Approach**: Playwright handles login & cookie capture only. Data fetching uses DeepSeek's internal API for speed and reliability.

### Install Dependencies & Import Libraries
Run this cell once to install required packages. Subsequent runs will skip installation if packages are already present.

In [1]:
# ── Install dependencies (skip if already installed) ──
import subprocess, sys

def install_if_missing(package, import_name=None):
    """Install a package only if it's not already importable."""
    import_name = import_name or package
    try:
        __import__(import_name)
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"✓ {package} installed")

install_if_missing("playwright")
install_if_missing("markdownify")
install_if_missing("requests")

# ── Install Playwright browser binaries (Chromium only) ──
print("\nInstalling Playwright Chromium browser...")
result = subprocess.run(
    [sys.executable, "-m", "playwright", "install", "chromium"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✓ Playwright Chromium browser ready")
else:
    print(f"⚠ Playwright browser install issue:\n{result.stderr}")


✓ playwright already installed
✓ markdownify already installed
✓ requests already installed

Installing Playwright Chromium browser...
✓ Playwright Chromium browser ready


In [2]:
# ── Import all required libraries ──
import json
import re
import time
import logging
from pathlib import Path
from datetime import datetime, timedelta
from urllib.parse import urljoin

import requests
import markdownify
from playwright.sync_api import sync_playwright

print("\n── All imports successful ──")
print(f"Python:      {sys.version.split()[0]}")
print(f"Requests:    {requests.__version__}")
print(f"Markdownify: installed ✓")


── All imports successful ──
Python:      3.12.10
Requests:    2.33.1
Markdownify: installed ✓


### Configuration Constants
Paths, URLs, retry settings, and other constants used throughout the notebook. Adjust as needed.

In [3]:
# ── Paths ──
BASE_DIR = Path("./")
EXPORT_DIR = BASE_DIR / "DeepSeek_Exports"
COOKIE_FILE = BASE_DIR / "deepseek_cookies.json"
MANIFEST_FILE = EXPORT_DIR / "conversations_manifest.json"
FAILURE_LOG = EXPORT_DIR / "failed_exports.log"

# ── DeepSeek URLs ──
DEEPSEEK_BASE_URL = "https://chat.deepseek.com"
DEEPSEEK_LOGIN_URL = "https://chat.deepseek.com/sign_in"
DEEPSEEK_API_BASE = "https://chat.deepseek.com/api/v0"

# ── Retry & Timing ──
MAX_RETRIES = 3
RETRY_DELAY_SEC = 2
REQUEST_DELAY_SEC = 1.0   # Politeness delay between API calls
LOGIN_TIMEOUT_SEC = 120   # How long to wait for manual Google login
PAGE_LOAD_TIMEOUT_MS = 30_000

# ── Logging ──
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("deepseek_export")

# ── Create export directory ──
EXPORT_DIR.mkdir(exist_ok=True)

print(f"Export dir:   {EXPORT_DIR.resolve()}")
print(f"Cookie file:  {COOKIE_FILE.resolve()}")
print(f"Manifest:     {MANIFEST_FILE.resolve()}")
print(f"Base URL:     {DEEPSEEK_BASE_URL}")
print(f"API base:     {DEEPSEEK_API_BASE}")

Export dir:   C:\Users\HELEN1822\OneDrive - Willis Towers Watson\Documents\Github\MindForge\DeepSeek_Exports
Cookie file:  C:\Users\HELEN1822\OneDrive - Willis Towers Watson\Documents\Github\MindForge\deepseek_cookies.json
Manifest:     C:\Users\HELEN1822\OneDrive - Willis Towers Watson\Documents\Github\MindForge\DeepSeek_Exports\conversations_manifest.json
Base URL:     https://chat.deepseek.com
API base:     https://chat.deepseek.com/api/v0


### Login & Cookie Management
- If `deepseek_cookies.json` exists and is valid, reuse saved session.
- Otherwise, opens a browser for manual Google login, then saves cookies for future runs.
- Also captures the `Authorization` bearer token from API requests for direct HTTP calls.

In [ ]:
# ── Session state (populated by login) ──
session_cookies = []   # Playwright cookies list
auth_token = None      # Bearer token for API calls
http_session = None    # requests.Session with auth headers

SCRIPTS_DIR = BASE_DIR / "scripts"


def save_cookies(cookies):
    """Save cookies to local JSON file."""
    with open(COOKIE_FILE, "w", encoding="utf-8") as f:
        json.dump(cookies, f, indent=2)
    log.info(f"Cookies saved to {COOKIE_FILE}")


def load_cookies():
    """Load cookies from local JSON file, or return None."""
    if not COOKIE_FILE.exists():
        return None
    with open(COOKIE_FILE, "r", encoding="utf-8") as f:
        cookies = json.load(f)
    log.info(f"Loaded {len(cookies)} cookies from {COOKIE_FILE}")
    return cookies


def login():
    """Run Playwright login in a subprocess (scripts/pw_login.py) to avoid Jupyter event loop conflicts."""
    global session_cookies, auth_token, http_session

    script_path = SCRIPTS_DIR / "pw_login.py"
    log.info("Launching Playwright login subprocess...")
    result = subprocess.run(
        [sys.executable, str(script_path),
         str(COOKIE_FILE), str(LOGIN_TIMEOUT_SEC), str(PAGE_LOAD_TIMEOUT_MS)],
        capture_output=True, text=True, timeout=LOGIN_TIMEOUT_SEC + 30
    )

    if result.returncode != 0:
        log.error(f"Login subprocess failed:\n{result.stderr}")
        raise RuntimeError(f"Login failed: {result.stderr[-500:]}")

    # Parse result from stdout
    token = None
    for line in result.stdout.splitlines():
        if line.startswith("RESULT:"):
            data = json.loads(line[7:])
            token = data.get("token")
        elif line == "WAITING_FOR_LOGIN":
            print("Browser opened — please log in with your Google account...")

    # Load cookies saved by subprocess
    session_cookies = load_cookies() or []
    auth_token = token

    # Build requests.Session
    http_session = requests.Session()
    for c in session_cookies:
        http_session.cookies.set(c["name"], c["value"], domain=c.get("domain", ""))
    if auth_token:
        http_session.headers["Authorization"] = f"Bearer {auth_token}"
        log.info("✓ Auth token captured")
    else:
        log.warning("⚠ No auth token captured — API calls may fail")

    http_session.headers["User-Agent"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    http_session.headers["Accept"] = "application/json"

    print(f"\n✓ Login complete")
    print(f"  Cookies: {len(session_cookies)}")
    print(f"  Token:   {'captured' if auth_token else 'MISSING'}")


# ── Run login ──
login()

08:48:36 [INFO] Launching Playwright login subprocess...
08:48:41 [ERROR] Login subprocess failed:
Traceback (most recent call last):
  File "c:\Users\HELEN1822\OneDrive - Willis Towers Watson\Documents\Github\MindForge\_pw_login_helper.py", line 84, in <module>
    result = manual_login(pw)
             ^^^^^^^^^^^^^^^^
  File "c:\Users\HELEN1822\OneDrive - Willis Towers Watson\Documents\Github\MindForge\_pw_login_helper.py", line 53, in manual_login
    page.goto(DEEPSEEK_LOGIN_URL, timeout=PAGE_TIMEOUT)
  File "c:\Users\HELEN1822\OneDrive - Willis Towers Watson\Documents\Github\MindForge\mindforge-env\Lib\site-packages\playwright\sync_api\_generated.py", line 9054, in goto
    self._sync(
  File "c:\Users\HELEN1822\OneDrive - Willis Towers Watson\Documents\Github\MindForge\mindforge-env\Lib\site-packages\playwright\_impl\_sync_base.py", line 115, in _sync
    return task.result()
           ^^^^^^^^^^^^^
  File "c:\Users\HELEN1822\OneDrive - Willis Towers Watson\Documents\Github\Min

RuntimeError: Login failed: ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\HELEN1822\OneDrive - Willis Towers Watson\Documents\Github\MindForge\mindforge-env\Lib\site-packages\playwright\_impl\_connection.py", line 559, in wrap_api_call
    raise rewrite_error(error, f"{parsed_st['apiName']}: {error}") from None
playwright._impl._errors.Error: Page.goto: net::ERR_SSL_VERSION_OR_CIPHER_MISMATCH at https://chat.deepseek.com/sign_in
Call log:
  - navigating to "https://chat.deepseek.com/sign_in", waiting until "load"



### Fetch All Conversations
Uses DeepSeek's internal API to fetch the conversation list. Falls back to DOM scraping via Playwright subprocess if the API doesn't work.

In [ ]:
def api_request(endpoint, method="GET", **kwargs):
    """Make an API request with retry logic."""
    url = f"{DEEPSEEK_API_BASE}/{endpoint.lstrip('/')}"
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = http_session.request(method, url, timeout=30, **kwargs)
            if resp.status_code == 200:
                return resp.json()
            elif resp.status_code == 401:
                log.error("Auth token expired — re-run the login cell")
                raise PermissionError("Auth token expired")
            else:
                log.warning(f"API {endpoint} returned {resp.status_code} (attempt {attempt}/{MAX_RETRIES})")
        except (requests.ConnectionError, requests.Timeout) as e:
            log.warning(f"API {endpoint} network error (attempt {attempt}/{MAX_RETRIES}): {e}")
        if attempt < MAX_RETRIES:
            time.sleep(RETRY_DELAY_SEC)
    raise RuntimeError(f"API {endpoint} failed after {MAX_RETRIES} attempts")


def parse_relative_date(date_str):
    """Convert relative dates like 'Today', 'Yesterday', '3 days ago', 'Apr 20' to YYYY-MM-DD."""
    today = datetime.now()
    s = date_str.strip().lower()

    if s == "today":
        return today.strftime("%Y-%m-%d")
    elif s == "yesterday":
        return (today - timedelta(days=1)).strftime("%Y-%m-%d")

    days_match = re.match(r"(\d+)\s+days?\s+ago", s)
    if days_match:
        return (today - timedelta(days=int(days_match.group(1)))).strftime("%Y-%m-%d")

    for fmt in ("%b %d", "%B %d"):
        try:
            parsed = datetime.strptime(s, fmt).replace(year=today.year)
            if parsed > today:
                parsed = parsed.replace(year=today.year - 1)
            return parsed.strftime("%Y-%m-%d")
        except ValueError:
            continue

    for fmt in ("%b %d, %Y", "%B %d, %Y", "%Y-%m-%d"):
        try:
            return datetime.strptime(s, fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue

    log.warning(f"Could not parse date: '{date_str}', using 'unknown'")
    return "unknown-date"


def fetch_conversations_api():
    """Fetch conversation list via DeepSeek's internal API."""
    log.info("Fetching conversations via API...")

    for endpoint in ["chat/list", "chat_list", "conversations", "chat/conversations"]:
        try:
            data = api_request(endpoint)
            if data and isinstance(data, dict):
                items = (data.get("data", {}).get("list") or
                         data.get("data", {}).get("chat_sessions") or
                         data.get("data", {}).get("conversations") or
                         data.get("data") if isinstance(data.get("data"), list) else
                         data.get("list") or
                         data.get("conversations") or
                         [])
                if items:
                    log.info(f"Found {len(items)} conversations via /{endpoint}")
                    conversations = []
                    for item in items:
                        conv_id = str(item.get("id") or item.get("chat_session_id") or item.get("conversation_id", ""))
                        title = item.get("title") or item.get("name") or "Untitled"
                        raw_date = (item.get("create_time") or item.get("created_at") or
                                    item.get("update_time") or item.get("updated_at") or "")
                        if raw_date and isinstance(raw_date, (int, float)):
                            date = datetime.fromtimestamp(raw_date).strftime("%Y-%m-%d")
                        elif raw_date:
                            date = parse_relative_date(str(raw_date))
                        else:
                            date = "unknown-date"
                        conversations.append({
                            "conversation_id": conv_id,
                            "original_title": title,
                            "date": date,
                            "url": f"{DEEPSEEK_BASE_URL}/a/{conv_id}",
                        })
                    return conversations
        except (RuntimeError, PermissionError):
            raise
        except Exception as e:
            log.debug(f"Endpoint /{endpoint} didn't work: {e}")
            continue

    return None


def fetch_conversations_fallback():
    """Fallback: use scripts/pw_fetch_convos.py to scroll sidebar and scrape conversation list."""
    log.info("API didn't return conversations. Falling back to DOM scraping...")

    script_path = SCRIPTS_DIR / "pw_fetch_convos.py"
    result = subprocess.run(
        [sys.executable, str(script_path), str(COOKIE_FILE), str(PAGE_LOAD_TIMEOUT_MS)],
        capture_output=True, text=True, timeout=120
    )

    if result.returncode != 0:
        raise RuntimeError(f"Fallback scraping failed:\n{result.stderr[-500:]}")

    for line in result.stdout.splitlines():
        if line.startswith("RESULT:"):
            convos = json.loads(line[7:])
            for c in convos:
                c.setdefault("date", "unknown-date")
            return convos

    raise RuntimeError("No result from fallback scraping")


def fetch_all_conversations():
    """Fetch all conversations — API first, DOM fallback second."""
    try:
        convos = fetch_conversations_api()
        if convos:
            return convos
    except PermissionError:
        raise
    except Exception as e:
        log.warning(f"API approach failed: {e}")

    return fetch_conversations_fallback()


# ── Fetch conversations ──
conversations = fetch_all_conversations()
print(f"\n✓ Found {len(conversations)} conversations")
for c in conversations[:5]:
    print(f"  [{c['date']}] {c['original_title'][:50]}  (id: {c['conversation_id'][:8]}...)")
if len(conversations) > 5:
    print(f"  ... and {len(conversations) - 5} more")

### Parse Single Conversation Messages
Fetches all messages for a given conversation via API (with DOM fallback). Skips thinking/reasoning tokens and image attachments. Preserves Markdown formatting.

In [ ]:
def fetch_messages_api(conversation_id):
    """Fetch messages for a conversation via the API."""
    for endpoint in [
        f"chat/{conversation_id}/messages",
        f"chat/messages?chat_session_id={conversation_id}",
        f"chat_messages/{conversation_id}",
        f"chat/{conversation_id}",
    ]:
        try:
            data = api_request(endpoint)
            if not data:
                continue

            messages_raw = (
                data.get("data", {}).get("messages") if isinstance(data.get("data"), dict) else
                data.get("data") if isinstance(data.get("data"), list) else
                data.get("messages") or
                data.get("data", {}).get("chat_messages") if isinstance(data.get("data"), dict) else
                []
            )

            if not messages_raw and isinstance(data.get("data"), dict):
                messages_raw = data["data"].get("chat_messages", [])

            if messages_raw:
                messages = []
                for msg in messages_raw:
                    role = msg.get("role", "").lower()
                    if role not in ("user", "assistant"):
                        continue
                    content = msg.get("content", "")
                    if not content.strip():
                        continue
                    if role == "assistant":
                        content = re.sub(r"<think>.*?</think>", "", content, flags=re.DOTALL).strip()
                        content = re.sub(r"<reasoning>.*?</reasoning>", "", content, flags=re.DOTALL).strip()
                    if content.strip():
                        messages.append({"role": role, "content": content.strip()})

                log.info(f"  Fetched {len(messages)} messages via API for {conversation_id[:8]}...")
                return messages
        except (RuntimeError, PermissionError):
            raise
        except Exception as e:
            log.debug(f"  Endpoint /{endpoint} failed: {e}")
            continue

    return None


def fetch_messages_fallback(conversation_id, conv_url):
    """Fallback: scrape messages using scripts/pw_fetch_messages.py."""
    log.info(f"  Falling back to DOM scraping for {conversation_id[:8]}...")

    script_path = SCRIPTS_DIR / "pw_fetch_messages.py"
    result = subprocess.run(
        [sys.executable, str(script_path), str(COOKIE_FILE), conv_url, str(PAGE_LOAD_TIMEOUT_MS)],
        capture_output=True, text=True, timeout=60
    )

    if result.returncode != 0:
        raise RuntimeError(f"DOM scraping failed for {conversation_id[:8]}: {result.stderr[-300:]}")

    for line in result.stdout.splitlines():
        if line.startswith("RESULT:"):
            raw_msgs = json.loads(line[7:])
            messages = []
            for msg in raw_msgs:
                html = msg.get("html", "")
                if html:
                    content = markdownify.markdownify(html, heading_style="ATX", strip=["img"])
                else:
                    content = msg.get("content", "")
                content = content.strip()
                if content:
                    messages.append({"role": msg["role"], "content": content})
            return messages

    raise RuntimeError(f"No result from DOM scraping for {conversation_id[:8]}")


def fetch_messages(conversation_id, conv_url):
    """Fetch messages for a conversation — API first, DOM fallback."""
    try:
        msgs = fetch_messages_api(conversation_id)
        if msgs:
            return msgs
    except PermissionError:
        raise
    except Exception as e:
        log.debug(f"  API failed for {conversation_id[:8]}: {e}")

    return fetch_messages_fallback(conversation_id, conv_url)


# ── Quick test (will only work when logged in and on open network) ──
print("✓ Message parsing functions defined")
print("  fetch_messages(conversation_id, url) → list of {role, content}")

### Generate Markdown Files & Update Manifest
Converts each conversation into an Obsidian-ready `.md` file with YAML frontmatter. Maintains `conversations_manifest.json` for incremental updates.

In [ ]:
def load_manifest():
    """Load existing manifest or return empty structure."""
    if MANIFEST_FILE.exists():
        with open(MANIFEST_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"last_updated": None, "conversations": []}


def save_manifest(manifest):
    """Save manifest to JSON file."""
    manifest["last_updated"] = datetime.now().isoformat()
    with open(MANIFEST_FILE, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)
    log.info(f"Manifest updated ({len(manifest['conversations'])} entries)")


def get_exported_ids(manifest):
    """Get set of already-exported conversation IDs from manifest."""
    return {c["conversation_id"] for c in manifest.get("conversations", [])}


def make_filename(date, conversation_id):
    """Generate filename: YYYY-MM-DD_first8chars.md"""
    short_id = conversation_id[:8]
    return f"{date}_{short_id}.md"


def generate_markdown(conversation, messages):
    """Generate Obsidian-compatible Markdown with YAML frontmatter."""
    title = conversation.get("original_title", "Untitled")
    date = conversation.get("date", "unknown-date")
    conv_id = conversation.get("conversation_id", "")
    url = conversation.get("url", "")

    # YAML frontmatter
    # Escape quotes in title for YAML safety
    safe_title = title.replace('"', '\\"')
    lines = [
        "---",
        f'date: "{date}"',
        f'original_title: "{safe_title}"',
        f'conversation_id: "{conv_id}"',
        f'url: "{url}"',
        "---",
        "",
        f"# {title}",
        "",
    ]

    # Conversation body
    for msg in messages:
        role = msg["role"]
        content = msg["content"]
        lines.append(f"**{role}**: {content}")
        lines.append("")

    return "\n".join(lines)


def export_conversation(conversation, manifest):
    """Export a single conversation to Markdown. Returns (success, filename_or_error)."""
    conv_id = conversation["conversation_id"]
    date = conversation["date"]
    url = conversation["url"]

    try:
        # Fetch messages
        messages = fetch_messages(conv_id, url)
        if not messages:
            return False, f"No messages found for {conv_id[:8]}"

        # Generate markdown
        md_content = generate_markdown(conversation, messages)

        # Write file
        filename = make_filename(date, conv_id)
        filepath = EXPORT_DIR / filename
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(md_content)

        # Update manifest
        manifest["conversations"].append({
            "conversation_id": conv_id,
            "original_title": conversation.get("original_title", "Untitled"),
            "date": date,
            "url": url,
            "file_name": filename,
            "exported_at": datetime.now().isoformat(),
        })

        log.info(f"  ✓ Exported: {filename}")
        return True, filename

    except Exception as e:
        log.error(f"  ✗ Failed {conv_id[:8]}: {e}")
        return False, str(e)


print("✓ Markdown generation functions defined")
print("  export_conversation(conversation, manifest) → (success, filename_or_error)")

### Main Workflow
Runs the full export pipeline: skips already-exported conversations (incremental), exports new ones, saves manifest, and logs failures.

In [ ]:
# ── Main export pipeline ──
manifest = load_manifest()
exported_ids = get_exported_ids(manifest)

# Filter to new conversations only
new_convos = [c for c in conversations if c["conversation_id"] not in exported_ids]
skipped = len(conversations) - len(new_convos)

print(f"Total conversations: {len(conversations)}")
print(f"Already exported:    {skipped}")
print(f"New to export:       {len(new_convos)}")
print()

successes = 0
failures = []

for i, conv in enumerate(new_convos, 1):
    print(f"[{i}/{len(new_convos)}] {conv['original_title'][:50]}...")
    success, result = export_conversation(conv, manifest)
    if success:
        successes += 1
    else:
        failures.append({"conversation_id": conv["conversation_id"],
                         "title": conv.get("original_title", ""),
                         "error": result})

    # Save manifest after each export (in case of crash)
    save_manifest(manifest)

    # Politeness delay between conversations
    if i < len(new_convos):
        time.sleep(REQUEST_DELAY_SEC)

# Write failure log
if failures:
    with open(FAILURE_LOG, "w", encoding="utf-8") as f:
        for fail in failures:
            f.write(f"{fail['conversation_id']} | {fail['title']} | {fail['error']}\n")
    log.warning(f"Wrote {len(failures)} failures to {FAILURE_LOG}")

print(f"\n{'=' * 50}")
print(f"Export complete!")
print(f"  ✓ Succeeded: {successes}")
print(f"  ✗ Failed:    {len(failures)}")
print(f"  ⊘ Skipped:   {skipped}")
print(f"  Output dir:  {EXPORT_DIR.resolve()}")

### Summary Statistics
Review exported files and manifest contents.

In [ ]:
# ── Summary ──
md_files = list(EXPORT_DIR.glob("*.md"))
manifest_data = load_manifest()

print(f"Markdown files in {EXPORT_DIR.resolve()}:")
print(f"  Total .md files: {len(md_files)}")
print()

# File size stats
if md_files:
    sizes = [f.stat().st_size for f in md_files]
    print(f"  Smallest: {min(sizes):,} bytes")
    print(f"  Largest:  {max(sizes):,} bytes")
    print(f"  Total:    {sum(sizes):,} bytes ({sum(sizes)/1024:.1f} KB)")
    print()

print(f"Manifest entries: {len(manifest_data.get('conversations', []))}")
print(f"Last updated:     {manifest_data.get('last_updated', 'never')}")

# Show failures if any
if FAILURE_LOG.exists():
    with open(FAILURE_LOG, "r", encoding="utf-8") as f:
        fail_lines = f.readlines()
    if fail_lines:
        print(f"\n⚠ {len(fail_lines)} failed exports (see {FAILURE_LOG}):")
        for line in fail_lines[:10]:
            print(f"  {line.strip()}")
else:
    print("\n✓ No failures logged")